In [1]:
R.<q,t,a> = LaurentPolynomialRing(QQ,3);
def Absolut(U):    #counts number of 1's in U
    A=0
    for x in U:
        if x==1:
            A=A+1
    return A
def HHH(U,V):      #recursion for HHH 
#    print(U)
#    print(V)
    if U[0]==0 and V[0]==0:
        U1=U[1:]
        U1.append(1)
        V1=V[1:]
        V1.append(1)
        U0=U[1:]
        U0.append(0)
        V0=V[1:]
        V0.append(0)
        return t^(-Absolut(U))*(HHH(U1,V1)+q*HHH(U0,V0))
    if U[0]==1 and V[0]==0:
        U1=U[1:]
        U1.append(1)
        return HHH(U1,V[1:])
    if U[0]==0 and V[0]==1:
        V1=V[1:]
        V1.append(1)
        return HHH(U[1:],V1)
    if U[0]==1 and V[0]==1:
        if len(U)>1 or len(V)>1:
            return (t^(Absolut(U)-1)+a)*HHH(U[1:],V[1:])
        else:
            return 1+a

def IsAdm(A,n,d):  # A is a subset of Z/ndZ; this tests if A is admissible: it is inadmissible if it contains all elements in a congruence class mod d.
    test=0
    for i in range(d):
        test=0
        for j in range(n):
            if d*j+i in A:
                test=test+1
            if test==n:
                return False
    return True
def Aminus(A,n,d): # A is a subset of Z/ndZ; this returns A-1 (mod nd)
    B=[]
    for i in A:
        if i==0:
            B.append(n*d-1)
        else:
            B.append(i-1)
    return B
def CJPoinc(U,V,A,n,m,d):  # recursion computing the (nd,md) q,t-Catalan polynomial via admissible nd,md - invariant subsets
    if not IsAdm(A,n,d):
        return 0
    test=True
    for i in U:
        if not i==2:
            test=False
    for i in V:
        if not i==2:
            test=False
    if test:
        return 1
    if U[0]==0 and V[0]==0:
        U1=U[1:]
        U1.append(1)
        V1=V[1:]
        V1.append(1)
        U0=U[1:]
        U0.append(0)
        V0=V[1:]
        V0.append(0)
        if U[len(U)-1]==2:
            B=Aminus(A,n,d)
            if not 0 in B:
                B.append(0)
            return t^(-Absolut(U))*(CJPoinc(U1,V1,Aminus(A,n,d),n,m,d)+q*CJPoinc(U0,V0,B,n,m,d))
        else:
            return t^(-Absolut(U))*(CJPoinc(U1,V1,Aminus(A,n,d),n,m,d)+q*CJPoinc(U0,V0,Aminus(A,n,d),n,m,d))
    if U[0]==1 and V[0]==0:
        U1=U[1:]
        U1.append(1)
        V2=V[1:]
        V2.append(2)
        return CJPoinc(U1,V2,Aminus(A,n,d),n,m,d)
    if U[0]==0 and V[0]==1:
        U2=U[1:]
        U2.append(2)
        V1=V[1:]
        V1.append(1)
        return CJPoinc(U2,V1,Aminus(A,n,d),n,m,d)
    if U[0]==1 and V[0]==1:
        U2=U[1:]
        U2.append(2)
        V2=V[1:]
        V2.append(2)
        return (t^(Absolut(U)-1)+a)*CJPoinc(U2,V2,Aminus(A,n,d),n,m,d)
    if U[0]==2 and V[0]==2:
        U2=U[1:]
        U2.append(2)
        V2=V[1:]
        V2.append(2)
        return CJPoinc(U2,V2,Aminus(A,n,d),n,m,d)
def PMax(n,m,d):  #returns maximal (or is it minimal?) nd,md Dyck path
    P=[]
    for i in range(n*d):
        P.append(floor(m*d-(i+1)*m*n^(-1)))
    return Partition(P)
def Ngens(P,n,m,d): # returns the set of rightmost boxes inside in rows of P (here P is treated as a lattice path crossing nd x Z strip, so this returns nd boxes)
    A=[]
    for i in range(n*d):
        if i<P.length():
            A.append((i,P[i]-1))
        else:
            A.append((i,-1))
    return A
def Mcogens(P,n,m,d): # return the set of the lowest boxes outside of P (here P is treated as a lattice path inside a Z x md strip, so this returns md boxes)
    A=[]
    if P.length()>0:
        for j in range(P[0],m*d):
            A.append((0,j))
        for i in range(P.length()-1):
            for j in range(P[i+1],P[i]):
                A.append((i+1,j))
        for j in range(P[P.length()-1]):
            A.append((P.length(),j))
    else:
        for j in range(m*d):
            A.append((0,j))
    return A
def Rank(A,n,m,d): # A is a box; the linear rank function normilized so that Rank(-1,md-1)=Rank(nd-1,-1)=0
    return n*m*d-n-m-A[0]*m-A[1]*n
def IsAbB(A,B,n,m,d): # A and B are boxes; answers to 'Is A > B according to rank with the first coordinate as tiebreaker' 
    if Rank(A,n,m,d)>Rank(B,n,m,d):
        return True
    elif Rank(A,n,m,d)<Rank(B,n,m,d):
        return False
    elif A[0]<B[0]:
        return True
    else:
        return False
def codinv(P,n,m,d): # Compute codinv(P) using the order on generators and cogenerators
    x=0
    A=Ngens(P,n,m,d)
    B=Mcogens(P,n,m,d)
    for a in A:
        for b in B:
            if IsAbB(b,a,n,m,d):
                x=x+1
    return x
def dinv(P,n,m,d): # Computes dinv using Haiman's leg-arm formula
    x=0
    for c in P.cells():
        if P.leg_length(c[0],c[1])*m < (P.arm_length(c[0],c[1])+1)*n and (P.leg_length(c[0],c[1])+1)*m >= P.arm_length(c[0],c[1])*n:
            x=x+1
    return x
def afact(P,n,m,d): # computes the factor for getting the higher a-power; Fixed!
    dcogens=P.addable_cells()
    B=Mcogens(P,n,m,d)
    f=1
    for g in dcogens:
        x=0
        for b in B:
            if IsAbB(b,g,n,m,d) and IsAbB((g[0],g[1]-1),b,n,m,d):
                x=x+1
        f=f*(1+a*t^(-x))
    return f
def qtCat(n,m,d): # computes the q,t-Catalan (with a=0); with general a computes the q,t - Schroeder polynomial 
    Max=PMax(n,m,d)
    delt=Max.size()
    f=0
    for s in range(delt+1):
        for P in Partitions(s,outer=Max):
            f=f+q^(delt-s)*t^dinv(P,n,m,d)*afact(P,n,m,d)
    return f
def CJinput(n,m,d): # generates (0001,000001) input for the СJPoinc recursion 
    A=[]
    B=[]
    for i in range(n*d-1):
        A.append(0)
    A.append(1)
    for i in range(m*d-1):
        B.append(0)
    B.append(1)
    return [A,B]
def HHHinput(n,m,d): # generates (0100,0000001) input for the HHH recursion
    A=[]
    B=[]
    for i in range(d-1):
        A.append(0)
    A.append(1)
    for i in range((n-1)*d):
        A.append(0)
    for i in range(m*d):
        B.append(0)
    B.append(1)
    return [A,B]
R=QQ['q','t','a']
q=R.0
t=R.1
a=R.2
# Definitions of coarm and coleg
def coarm(tab,i):
    return tab.cells_containing(i)[0][0]
def coleg(tab,i):
    return tab.cells_containing(i)[0][1]
# (q,t)-contents of boxes of a skew tableau
def zz(tab):
    size=tab.size()
    return [q^coarm(tab,i)*t^coleg(tab,i) for i in range(1,size+1)]
# definition of omega
def std(x):
    if x==0: 
        return 1
    else:
        return x
def om(x):
    return std(1-x)*std(1-q*t*x)/(std(1-q*x)*std(1-t*x))
# weight for a fixed point in FHilb(line)
def weight(z):
    size=len(z)
    res=prod([om(z[i]/z[j])for i in range(size) for j in range(i+1,size)])
    res=res/prod([std(1-1/z[i]) for i in range(size)])
    res=res/(1-q)^(size)
    return factor(res)
# weight for a fixed point in FHilb(point)
def weightpt(z):
    size=len(z)
    res=prod([om(z[i]/z[j])for i in range(size) for j in range(i+1,size)])
    res=res/prod([std(1-1/z[i]) for i in range(size)])
    res=res/prod([(1-q*t*z[i-1]/z[i]) for i in range(1,size)])
    return res
# weight for a fixed point in Z(line)
def weightA(z):
    size=len(z)
    res=prod([om(z[i]/z[j])for i in range(size) for j in range(i+1,size)])
    res=res/prod([std(1-1/z[i]) for i in range(size)])
    res=res*prod([(1+a/z[i]) for i in range(size)])
    res=res/(1-q)^(size)
    return factor(res)
# weight for a fixed point in Z(pt)
def weightApt(z):
    size=len(z)
    res=prod([om(z[i]/z[j])for i in range(size) for j in range(i+1,size)])
    res=res/prod([std(1-1/z[i]) for i in range(size)])
    res=res*prod([(1+a/z[i]) for i in range(1,size)])
    res=res/prod([(1-q*t*z[i-1]/z[i]) for i in range(1,size)])
    return factor(res)
# weight for a line bundle
def bdle(z,po):
    size=len(z)
    res=prod([z[i]^(po[i]) for i in range(size)])
    return res
# Euler characteristic
def chi(po,opt):
    size=len(po)
    lst=StandardTableaux(size).list()
    if opt=="line":
        return sum([weight(zz(tab))*bdle(zz(tab),po) for tab in lst])
    if opt=="point":
        return sum([weightpt(zz(tab))*bdle(zz(tab),po) for tab in lst])
    if opt=="full":
        return sum([weightA(zz(tab))*bdle(zz(tab),po) for tab in lst])
    if opt=="fullpt":
        return sum([weightApt(zz(tab))*bdle(zz(tab),po) for tab in lst])
def chitest(po,j):
    size=len(po)
    lst=StandardTableaux(size).list()
    return sum([weight(zz(tab))*bdle(zz(tab),po)*(1-q)/(1-q*t*zz(tab)[j]/zz(tab)[j+1]) for tab in lst])
def chisimp(po):
    tmp=chi(po,"point")
    return tmp.subs({q:1,t:1})

#
#
#

def Is_alm_const(d): #checks if the sequence d varies by at most one except for the last, which can be smaller
    l=len(d)
    if max(d[:l-1])-min(d[:l-1])>1:
        return 0
    else: 
        if d[l-1] > min(d[:l-1])+1:
            return 0
        else:
            return 1

def Is_triangular(P): #checks if the partition P is triangular
    l=P.length()
    d=[]
    for i in range(l-1):
        d.append(P[i]-P[i+1])
    d.append(P[l-1])
    for k in range(1,l):
        dd=[]
        for i in range(l-k+1):
            dd.append(0)
            for j in range(k):
                dd[i]=dd[i]+d[i+j]
        if Is_alm_const(dd)==0:
            return 0
    return 1
def Is_alm_decr(d): #checks if the sequence d doesn't increase by more than one
    l=len(d)
    for i in range(l-1):
        if max(d[i+1:])-d[i]>1:
            return 0
    return 1
def Is_concave(P): #checks if the partition P is concave
    l=P.length()
    d=[]
    for i in range(l-1):
        d.append(P[i]-P[i+1])
    d.append(P[l-1])
    for k in range(1,l):
        dd=[]
        for i in range(l-k+1):
            dd.append(0)
            for j in range(k):
                dd[i]=dd[i]+d[i+j]
        if Is_alm_decr(dd)==0:
            return 0
    return 1
def max_min_slope(P): #finds the extreme slopes of a traingular partition P
    minsl=[0,1]
    maxsl=[1,0]
    outs=P.outside_corners()
    ins=P.corners()
    for outcor in P.outside_corners():
        for incor in P.corners():
            if outcor[0] > incor[0] and outcor[1] < incor[1]:
                if maxsl[1]==0:
                    maxsl=[outcor[0]-incor[0],incor[1]-outcor[1]]
                elif (outcor[0]-incor[0])/(incor[1]-outcor[1]) < maxsl[0]/maxsl[1]:
                    maxsl=[outcor[0]-incor[0],incor[1]-outcor[1]]
            if outcor[0] < incor[0] and outcor[1] > incor[1]:
                if minsl[0]==0:
                    minsl=[incor[0]-outcor[0],outcor[1]-incor[1]]
                elif (incor[0]-outcor[0])/(outcor[1]-incor[1]) > minsl[0]/minsl[1]:
                    minsl=[incor[0]-outcor[0],outcor[1]-incor[1]]
    maxred=[Integer(maxsl[0]/math.gcd(maxsl[0],maxsl[1])),Integer(maxsl[1]/math.gcd(maxsl[0],maxsl[1]))]
    minred=[Integer(minsl[0]/math.gcd(minsl[0],minsl[1])),Integer(minsl[1]/math.gcd(minsl[0],minsl[1]))]
    return [maxred,minred]
def slope(P): #picks a good slope (n,m) of the form (k*maxsl+red(minsl+maxsl))
    temp=max_min_slope(P)
    maxsl=temp[0]
    minsl=temp[1]
    sumsl=[maxsl[0]+minsl[0],maxsl[1]+minsl[1]]
    d=math.gcd(sumsl[0],sumsl[1])
    midsl=[Integer(sumsl[0]/d),Integer(sumsl[1]/d)]
    #print(midsl)
    goodsl=[midsl[0]+maxsl[0],midsl[1]+maxsl[1]]
    while ((goodsl[0]-1)*(goodsl[1]-1) < 2*P.size()):
        #print(goodsl)
        goodsl=[goodsl[0]+maxsl[0],goodsl[1]+maxsl[1]]
    return goodsl
def binseq(P): #return the binary sequences corresponding to a triangular partition
    sl=slope(P)
    PP=P.conjugate()
    a=[] #sequence of -Andersen labels of the tops of columns
    b=[] #sequence of -Andersen labels of the ends of rows
    for i in range(P[0]):
        a.append((PP[i]-1)*sl[1]+i*sl[0])
    M=max(a) #fixing the maximum for -Andresen under the diagonal <--> fixing the diagonal
    if -sl[1]+P[0]*sl[0] < M:
        a.append(-sl[1]+P[0]*sl[0])
    t=(PP[0]-1)*sl[1]-sl[0]
    while t+sl[1]<M:
        t=t+sl[1]
    a.insert(0,t)
    while a[len(a)-1]+sl[0] < M:
        a.append(a[len(a)-1]+sl[0])
    
    for i in range(PP[0]):
        b.append((P[i]-1)*sl[0]+i*sl[1])
    if -sl[0]+PP[0]*sl[1] < M:
        b.append(-sl[0]+PP[0]*sl[1])
    t=(P[0]-1)*sl[0]-sl[1]
    while t+sl[0]<M:
        t=t+sl[0]
    b.insert(0,t)
    while b[len(b)-1]+sl[1] < M:
        b.append(b[len(b)-1]+sl[1])
    u=[1]
    v=[1]
    for i in range(1,len(a)):
        if a[i] > a[0]:
            u.insert(0,0)
        else:
            u.append(0)
    for i in range(1,len(b)):
        if b[i] > b[0]:
            v.insert(0,0)
        else:
            v.append(0)
    return [u,v]
def Insert_strand(u,v,k): #inserts a strand into the diagram of the pair if binary sequences (u,v), starting at the end of v and adding k elements to u
    p=len(v)
    v.append(1)
    #print(p,u,v,"v")
    i=0
    while i < k:
        while p > len(u):
            zz=p-len(u)
            p=0
            while zz>0:
                if v[p]==0:
                    zz=zz-1
                    p=p+1
                else:
                    p+p+1
            v.insert(p,0)
            #print(p,u,v,"v")
        if i < k-1:
            u.insert(p,0)
            i=i+1
            #print(p,u,v,"u",i)
            zz=0
            for j in range(p+1,len(u)):
                if(u[j]==0):
                    zz=zz+1
            while zz > len(v)-sum(v) and i < k:
                zz=zz-(len(v)-sum(v))
                p=len(u)
                zzz=zz
                while zzz>0:
                    if u[p-1]==0:
                        zzz=zzz-1
                        p=p-1
                    else:
                        p=p-1
                if i < k-1:
                    u.insert(p,0)
                    i=i+1
                    #print(p,u,v,"u",i)
                elif i==k-1:
                    u.insert(p,1)
                    i=i+1
                    #print(p,u,v,"u",i)
            p=len(v)
            while zz > 0 and i < k:
                if v[p-1]==0:
                    p=p-1
                    zz=zz-1
                else:
                    p=p-1
            if i < k:
                v.insert(p,0)
                #print(p,u,v,"v")
        elif i==k-1:
            u.insert(p,1)
            i=i+1
            #print(p,u,v,"u",i)
    return [u,v]
def triang(P): #for a concave P returns the maximal triangular subpartition with the same first part
    if Is_triangular(P)==1:
        return P
    check=0
    l=P.length()
    if l < 3:
        check=1
        PP=P
        k=l
    d=[]
    for i in range(l-1):
        d.append(P[i]-P[i+1])
    d.append(P[l-1])
    k=3
    while k < l and check==0:
        Pl=[]
        for i in range(k):
            Pl.append(P[i]-P[k])
        PP=Partition(Pl)
        if Is_triangular(PP)==1:
            k=k+1
        else:
            k=k-1
            Pl=[]
            for i in range(k):
                Pl.append(P[i]-P[k])
            PP=Partition(Pl)
            check=1
    if k==l and check==0:
        k=k-1
        Pl=[]
        for i in range(k):
            Pl.append(P[i]-P[k])
        PP=Partition(Pl)
        check=1
    [n,m]=max_min_slope(PP)[0]
    M=0
    for box in PP.cells():
        temp=m*box[0]+n*box[1]
        if temp>M:
            M=temp
    M=M+P[k]*n
    Pl=[]
    for i in range(k):
        Pl.append(P[i])
    i=k
    while i*m < M:
        Pl.append(math.floor((M-i*m)/n))
        i=i+1
    PP=Partition(Pl)
    return PP

def Aseq(P): #returns the sequence of horizontal steps of a partition P
    l=P.length()
    A=[0]
    for i in range(l):
        if i==l-1:
            A.append(P[l-1])
        else:
            A.append(P[i]-P[i+1])
    return A




In [37]:
A=HHH([1,0,0,0,0],[0,0,0,1]) # applying HM recursion
B=HHH([0,1,0,1,0],[0,0,1,1]) # applying HM recursion

f=q*t^(-1)*A+t^(-1)*B
print(f)

f=t^(5)*f/(a+1) # renormalize
print(f)

g=chi([0,2,0,1],"fullpt") # computing EHA Schroder polynomial

print(f==g)

(q^5*a + q^4*t*a + q^3*t^2*a + q^2*t^3*a + q*t^4*a + t^5*a + q^4*a^2 + q^3*t*a^2 + q^2*t^2*a^2 + q*t^3*a^2 + t^4*a^2 + q^5 + q^4*t + q^3*t^2 + q^2*t^3 + q*t^4 + t^5 + q^4*a + 2*q^3*t*a + 3*q^2*t^2*a + 2*q*t^3*a + t^4*a + q^3*a^2 + 3*q^2*t*a^2 + 3*q*t^2*a^2 + t^3*a^2 + q^2*a^3 + 2*q*t*a^3 + t^2*a^3 + q^3*t + 2*q^2*t^2 + q*t^3 + q^3*a + 3*q^2*t*a + 3*q*t^2*a + t^3*a + q^2*a^2 + 2*q*t*a^2 + t^2*a^2)/t^5
q^5 + q^4*t + q^3*t^2 + q^2*t^3 + q*t^4 + t^5 + q^4*a + q^3*t*a + q^2*t^2*a + q*t^3*a + t^4*a + q^3*t + 2*q^2*t^2 + q*t^3 + q^3*a + 3*q^2*t*a + 3*q*t^2*a + t^3*a + q^2*a^2 + 2*q*t*a^2 + t^2*a^2
True


In [38]:
# first we setup the binary sequences

A=Insert_strand([0,0,1,0,0],[0,0,0,1],1)
A=Insert_strand(A[0],A[1],1)
print(A)
B=Insert_strand([0,0,1,0,0],[0,0,0,1],2)
print(B)
C=Insert_strand([0,1,0,0,0,0],[0,0,0,0,1],1)
print(C)
D=[[0,0,0,0,0,0,1],[0,0,0,0,1]]
print(D)

# applying HM recursion and combining the answers

f=t^(-3)*HHH(A[0],A[1])+q*t^(-3)*HHH(B[0],B[1])+q*t^(-2)*HHH(C[0],C[1])+q^2*t^(-2)*HHH(D[0],D[1])
f=t^(14)*f/(a+1)
print(f)

# computing the EHA Schroder polynomial and checking

g=chi([0,1,2,0,1,1],"fullpt")
print(f==g)

[[0, 0, 1, 0, 1, 1, 0], [0, 0, 0, 1, 1, 1]]
[[0, 0, 1, 1, 0, 0, 0], [0, 0, 0, 0, 1, 1]]
[[0, 1, 0, 0, 0, 1, 0], [0, 0, 0, 0, 1, 1]]
[[0, 0, 0, 0, 0, 0, 1], [0, 0, 0, 0, 1]]
q^14 + q^13*t + q^12*t^2 + q^11*t^3 + q^10*t^4 + q^9*t^5 + q^8*t^6 + q^7*t^7 + q^6*t^8 + q^5*t^9 + q^4*t^10 + q^3*t^11 + q^2*t^12 + q*t^13 + t^14 + q^13*a + q^12*t*a + q^11*t^2*a + q^10*t^3*a + q^9*t^4*a + q^8*t^5*a + q^7*t^6*a + q^6*t^7*a + q^5*t^8*a + q^4*t^9*a + q^3*t^10*a + q^2*t^11*a + q*t^12*a + t^13*a + q^12*t + q^11*t^2 + q^10*t^3 + q^9*t^4 + q^8*t^5 + q^7*t^6 + q^6*t^7 + q^5*t^8 + q^4*t^9 + q^3*t^10 + q^2*t^11 + q*t^12 + q^12*a + 2*q^11*t*a + 2*q^10*t^2*a + 2*q^9*t^3*a + 2*q^8*t^4*a + 2*q^7*t^5*a + 2*q^6*t^6*a + 2*q^5*t^7*a + 2*q^4*t^8*a + 2*q^3*t^9*a + 2*q^2*t^10*a + 2*q*t^11*a + t^12*a + q^11*a^2 + q^10*t*a^2 + q^9*t^2*a^2 + q^8*t^3*a^2 + q^7*t^4*a^2 + q^6*t^5*a^2 + q^5*t^6*a^2 + q^4*t^7*a^2 + q^3*t^8*a^2 + q^2*t^9*a^2 + q*t^10*a^2 + t^11*a^2 + q^11*t + 2*q^10*t^2 + 2*q^9*t^3 + 2*q^8*t^4 + 2*q^7*t^5 + 2*q

In [39]:
M=[6,4,1] # any triangular partition
P=Partition(M)
l=P.length()
n=P.size()
N=[]
for i in range(l):
    N.append(M[i]-1)
Pp=Partition(N) # removing the first column

A=binseq(P) # computing the corresponding binary sequences
B=binseq(Pp) # computing the corresponding binary sequences
print(A)
print(B)

B=Insert_strand(B[0],B[1],1) # inserting an extra strand into B
print(B)

fA=HHH(A[0],A[1]) # applying HM recursion
fB=HHH(B[0],B[1]) # applying HM recursion
#print(fA)
#print(fB)

for k in range(1,5):
    f=t^(n+k)*((t^(-1)*fB*(1-q^k*t^(-k)))/(1-q*t^(-1))+q^k*t^(-k)*fA)/(a+1)
    M.append(1)
    Seq=Aseq(Partition(M)) # computing the sequence of horizontal steps
    print(Seq)
    g=chi(Seq,"fullpt") # computing the EHA Schroder polynomial
    print(f==g)

[[0, 0, 0, 0, 0, 0, 0, 1, 0], [0, 0, 1, 0]]
[[0, 0, 0, 1, 0, 0, 0, 0], [0, 0, 1, 0]]
[[0, 0, 0, 1, 1, 0, 0, 0, 0], [0, 0, 1, 0, 1]]
[0, 2, 3, 0, 1]
True
[0, 2, 3, 0, 0, 1]
True
[0, 2, 3, 0, 0, 0, 1]
True
[0, 2, 3, 0, 0, 0, 0, 1]
True


In [40]:
M=[6,4,1] # any triangular partition
P=Partition(M)
l=P.length()
n=P.size()
N=[]
for i in range(l):
    N.append(M[i]-1)
Pp=Partition(N) # removing the first column

A=binseq(P) # computing the corresponding binary sequences
B=binseq(Pp) # computing the corresponding binary sequences
print(A)
print(B)
B=Insert_strand(B[0],B[1],1) # inserting an extra strand into B
print(B)
fA=HHH(A[0],A[1]) # applying HM recursion
fB=HHH(B[0],B[1]) # applying HM recursion
#print(fA)
#print(fB)
for k in range(1,10):
    f=t^(n+k)*((t^(-1)*fB*(1-q^k*t^(-k)))/(1-q*t^(-1))+q^k*t^(-k)*fA)/(a+1)
    M.append(1)
    Seq=Aseq(Partition(M).conjugate()) # cheating by applying conjugation
    print(Seq)
    g=chi(Seq,"fullpt")
    print(f==g)

[[0, 0, 0, 0, 0, 0, 0, 1, 0], [0, 0, 1, 0]]
[[0, 0, 0, 1, 0, 0, 0, 0], [0, 0, 1, 0]]
[[0, 0, 0, 1, 1, 0, 0, 0, 0], [0, 0, 1, 0, 1]]
[0, 2, 0, 0, 1, 0, 1]
True
[0, 3, 0, 0, 1, 0, 1]
True
[0, 4, 0, 0, 1, 0, 1]
True
[0, 5, 0, 0, 1, 0, 1]
True
[0, 6, 0, 0, 1, 0, 1]
True
[0, 7, 0, 0, 1, 0, 1]
True
[0, 8, 0, 0, 1, 0, 1]
True
[0, 9, 0, 0, 1, 0, 1]
True
[0, 10, 0, 0, 1, 0, 1]
True


In [41]:
        
#f=t^(30)*HHH(A[0],A[1])/(a+1)
#As=Aseq(P)
#print(As)
#g=chi(Aseq(P),"fullpt")
#print(f)
#print(g)
#print(f==g)
#u=[0,0,0,1,0,0,0,0,0,0,0,0]
#v=[0,0,1,0,0]
#[u,v]=Insert_strand(u,v,1)
#Insert_strand(u,v,7)
#P=Partition([7,4,2])
#print(P.outside_corners())
#print(max_min_slope(P))
#print(slope(P))
#print(binseq(P))
#print(P.corners())
#print(P.outside_corners())
#part_list=[]
#for i in range(10,11):
#    part_list=part_list + Partitions(i).list()
#for P in part_list:
#    if Is_triangular(P)==1:
#        print(P)
#        print(max_min_slope(P))
#    print(Is_triangular(P))
#    print(Is_concave(P))

In [66]:
chi([0,3,1,2,0,1],"point")
for k in range(1,50):
    Fun=chi([0,3+k,1+k,2+k,k,1+k],"point").numerator()
    for c in Fun.coefficients():
        temp=0
        if c<0:
            temp=temp+1
            print("Negative coefficient found")
    if temp==0:
        print("All coefficients are non-negative")

All coefficients are non-negative
All coefficients are non-negative
All coefficients are non-negative
All coefficients are non-negative
All coefficients are non-negative
All coefficients are non-negative
All coefficients are non-negative
All coefficients are non-negative
All coefficients are non-negative
All coefficients are non-negative
All coefficients are non-negative
All coefficients are non-negative
All coefficients are non-negative
All coefficients are non-negative
All coefficients are non-negative
All coefficients are non-negative
All coefficients are non-negative
All coefficients are non-negative
All coefficients are non-negative
All coefficients are non-negative
All coefficients are non-negative
All coefficients are non-negative
All coefficients are non-negative
All coefficients are non-negative
All coefficients are non-negative
All coefficients are non-negative
All coefficients are non-negative
All coefficients are non-negative
All coefficients are non-negative
All coefficien

In [9]:
M=[4,3,2] 
P=Partition(M)
l=P.length()
n=P.size()

A=binseq(P) 

print(A)

A=Insert_strand(A[0],A[1],1)
A=Insert_strand(A[0],A[1],1)
A=Insert_strand(A[0],A[1],1)

print(A)

fA=t^(-6)*HHH(A[0],A[1])

B=binseq(P) 

B=Insert_strand(B[0],B[1],1)
B=Insert_strand(B[0],B[1],2)

print(B)

fB=q*t^(-6)*HHH(B[0],B[1])

C=binseq(P) 

C=Insert_strand(C[0],C[1],2)
C=Insert_strand(C[0],C[1],1)

print(C)

fC=q*t^(-5)*HHH(C[0],C[1])

D=binseq(P) 

D=Insert_strand(D[0],D[1],3)

print(D)

fD=q^(2)*t^(-5)*HHH(D[0],D[1])

M=[5,4,3,1] 
P=Partition(M)
l=P.length()
n=P.size()

E=binseq(P) 

print(E)

E=Insert_strand(E[0],E[1],1)
E=Insert_strand(E[0],E[1],1)

print(E)

fE=q*t^(-4)*HHH(E[0],E[1])

F=binseq(P) 

F=Insert_strand(F[0],F[1],2)

print(F)

fF=q^(2)*t^(-4)*HHH(F[0],F[1])

M=[6,5,4,2,1] 
P=Partition(M)
l=P.length()
n=P.size()

G=binseq(P) 

print(G)

G=Insert_strand(G[0],G[1],1)

print(G)

fG=q^(2)*t^(-3)*HHH(G[0],G[1])

M=[7,6,5,3,2,1] 
P=Partition(M)
l=P.length()
n=P.size()

H=binseq(P) 

print(H)

fH=q^(3)*t^(-3)*HHH(H[0],H[1])

f=t^(27)*(fA+fB+fC+fD+fE+fF+fG+fH)/(a+1)

M=[7,6,5,3,3,2,1] 

#Seq=Aseq(Partition(M)) # computing the sequence of horizontal steps
#g=chi(Seq,"fullpt") # computing the EHA Schroder polynomial
print(f)
print(t^(27)*fA/(a+1))
print(t^(27)*fB/(a+1))
print(t^(27)*fC/(a+1))
print(t^(27)*fD/(a+1))
print(t^(27)*fE/(a+1))
print(t^(27)*fF/(a+1))
print(t^(27)*fG/(a+1))
print(t^(27)*fH/(a+1))

[[0, 0, 0, 0, 1, 0], [0, 0, 0, 1, 0]]
[[0, 0, 0, 0, 1, 1, 1, 1, 0], [0, 0, 0, 1, 0, 1, 1, 1]]
[[0, 0, 0, 0, 1, 1, 1, 0, 0], [0, 0, 0, 1, 0, 0, 1, 1]]
[[0, 0, 0, 0, 1, 1, 0, 1, 0], [0, 0, 0, 1, 0, 0, 1, 1]]
[[0, 0, 0, 0, 1, 0, 1, 0, 0], [0, 0, 0, 1, 0, 0, 0, 1]]
[[0, 0, 0, 0, 1, 0, 0], [0, 0, 0, 1, 0, 0]]
[[0, 0, 0, 0, 1, 0, 1, 1, 0], [0, 0, 0, 1, 0, 0, 1, 1]]
[[0, 0, 0, 0, 1, 1, 0, 0, 0], [0, 0, 0, 1, 0, 0, 0, 1]]
[[0, 0, 0, 0, 1, 0, 0, 0], [0, 0, 0, 1, 0, 0, 0]]
[[0, 0, 0, 0, 1, 0, 0, 1, 0], [0, 0, 0, 1, 0, 0, 0, 1]]
[[0, 0, 0, 0, 1, 0, 0, 0, 0], [0, 0, 0, 1, 0, 0, 0, 0]]
q^27 + q^26*t + q^25*t^2 + q^24*t^3 + q^23*t^4 + q^22*t^5 + q^21*t^6 + q^20*t^7 + q^19*t^8 + q^18*t^9 + q^17*t^10 + q^16*t^11 + q^15*t^12 + q^14*t^13 + q^13*t^14 + q^12*t^15 + q^11*t^16 + q^10*t^17 + q^9*t^18 + q^8*t^19 + q^7*t^20 + q^6*t^21 + q^5*t^22 + q^4*t^23 + q^3*t^24 + q^2*t^25 + q*t^26 + t^27 + q^26*a + q^25*t*a + q^24*t^2*a + q^23*t^3*a + q^22*t^4*a + q^21*t^5*a + q^20*t^6*a + q^19*t^7*a + q^18*t^8*a + q^17*

In [10]:
M=[5,4,3,1] 
P=Partition(M)
l=P.length()
n=P.size()

E=binseq(P) 

print(E)

E=Insert_strand(E[0],E[1],1)
E=Insert_strand(E[0],E[1],1)

print(E)

fE=t^(-3)*HHH(E[0],E[1])

F=binseq(P) 

F=Insert_strand(F[0],F[1],2)

print(F)

fF=q*t^(-3)*HHH(F[0],F[1])

M=[6,5,4,2,1] 
P=Partition(M)
l=P.length()
n=P.size()

G=binseq(P) 

print(G)

G=Insert_strand(G[0],G[1],1)

print(G)

fG=q*t^(-2)*HHH(G[0],G[1])

M=[7,6,5,3,2,1] 
P=Partition(M)
l=P.length()
n=P.size()

H=binseq(P) 

print(H)

fH=q^(2)*t^(-2)*HHH(H[0],H[1])

f=t^(26)*(fE+fF+fG+fH)/(a+1)

M=[7,6,5,3,2,2,1] 

Seq=Aseq(Partition(M)) # computing the sequence of horizontal steps
g=chi(Seq,"fullpt") # computing the EHA Schroder polynomial
print(f==g)

[[0, 0, 0, 0, 1, 0, 0], [0, 0, 0, 1, 0, 0]]
[[0, 0, 0, 0, 1, 0, 1, 1, 0], [0, 0, 0, 1, 0, 0, 1, 1]]
[[0, 0, 0, 0, 1, 1, 0, 0, 0], [0, 0, 0, 1, 0, 0, 0, 1]]
[[0, 0, 0, 0, 1, 0, 0, 0], [0, 0, 0, 1, 0, 0, 0]]
[[0, 0, 0, 0, 1, 0, 0, 1, 0], [0, 0, 0, 1, 0, 0, 0, 1]]
[[0, 0, 0, 0, 1, 0, 0, 0, 0], [0, 0, 0, 1, 0, 0, 0, 0]]
True
